04-search_evaluation

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv") #-data.csv")

In [2]:
df_ground_truth.head()

,question,document
0,"I just found this course, is it still okay to ...",74eb249bbf
1,"If I start the course late, can I still get a ...",74eb249bbf
2,What’s the deadline for turning in the project...,74eb249bbf
3,"Can new students still participate, or is it t...",74eb249bbf
4,Is it possible to earn the certificate after s...,74eb249bbf


In [3]:
ground_truth = df_ground_truth.to_dict(orient="records")


In [4]:
ground_truth[10]

{'question': 'Where do I join the office hours or live workshop if I’m a student and don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [5]:
#Just rebuilding and indexing the faq data set, which we will be sending to the LLM,
#along with the ground truth data set, which we will be using to evaluate the LLM's performance.
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': 'db78580409',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity between two non-zero vectors, often used in text analysis to determine how similar two documents are based on their content. This metric computes the cosine of the angle between two vectors, which are typically word counts or TF-IDF values of the documents. The cosine similarity value ranges from -1 to 1, where 1 indicates that the vectors are identical, 0 indicates that the vectors are orthogonal (no similarity), and -1 represents completely opposite vectors.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](h

In [7]:
#Just wrapping the above codes into a function,
#where we set default boost values: "boost_dict = {"question": 3.0, "section": 0.5}" 
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

....Test "text_search" with our first question from ground_truth...

In [8]:
q = ground_truth[0]
q

{'question': 'I just found this course, is it still okay to join now?',
 'document': '74eb249bbf'}

....get the doc Id of the first question...

In [9]:
doc_id = q['document']
doc_id

'74eb249bbf'

....use text_search to ask the question...

In [10]:
results = text_search(q['question'])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in o

...and see if text_search returned the correct document!!

In [11]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False
0fab61eca2 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False


....Calculate the "relevance matrix", which can be used to evaluate how good the search is!

In [12]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [13]:
# Wrap the above logic into a function:
def compute_relevance_text(q):
    doc_id = q["document"] # Get id of the correct document for the question
    results = text_search(query=q["question"]) # Get text_search to search for the question and return the top 5 results

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [14]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course, is it still okay to join now?


[1, 0, 0, 0, 0]

In [15]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [0, 0, 1, 0, 0]

Will the live session video link be shared anywhere before the workshop starts?


[1, 0, 0, 0, 0]

In [16]:
#Imagine if we get this "relevance" result....we did find it, but it's in the 3rd position.
#which is not so good
[0, 0, 1, 0, 0]

[0, 0, 1, 0, 0]

In [17]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [0, 0, 0, 0, 0]

Where do I check the LLM Zoomcamp syllabus and homework deadlines?


[1, 0, 0, 0, 0]

In [18]:
# if we get all 0s in the relevance result, it means we cant find find any match!
[0, 0, 0, 0, 0]

[0, 0, 0, 0, 0]

In [19]:
#Compute the relevance for all questions in the ground_truth data set
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [20]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/565 [00:00<?, ?it/s]

In [21]:
relevance[:15]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0]]

In [22]:
# Making the search function generic, by passing it as an argument, 
# so we can pass any search function to it
# and this way we can evaluate the relevance value of different search functions, not just the text_search function.
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [23]:
# Making the search function generic, by passing it as an argument, 
# so we can pass any search function to it
# and this way we can evaluate the relevance value of different search functions, not just the text_search function.
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [24]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

05-search_metrics

In [25]:
sample = relevance_total[:15]

In [26]:
sample

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0]]

In [27]:
14 / 15

0.9333333333333333

In [28]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

0.8666666666666667

In [29]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [30]:
hit_rate(relevance)

0.863716814159292

In [31]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

0.6855555555555556

In [32]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)
    

In [33]:
mrr(sample)

0.6855555555555556

In [34]:
mrr(relevance) #relevance = the relevance score for the entire ground_truth data set

0.744837758112094

In [35]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [36]:
evaluate(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.863716814159292, 'mrr': 0.744837758112094}

06-search-tuning

In [37]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [38]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8831858407079646, 'mrr': 0.7654277286135688}

In [39]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [40]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8902654867256637, 'mrr': 0.7921533923303832}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8991150442477877, 'mrr': 0.8005899705014744}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.863716814159292, 'mrr': 0.744837758112094}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8424778761061947, 'mrr': 0.7062536873156338}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8088495575221238, 'mrr': 0.6781415929203536}


In [41]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [42]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

In [43]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
6,1.0,4.0,0.1,0.975221,0.872419
7,1.0,4.0,0.2,0.968142,0.871947
18,2.0,4.0,0.1,0.976991,0.870855
33,5.0,10.0,0.1,0.978761,0.870354
34,5.0,10.0,0.2,0.976991,0.870059
23,2.0,10.0,0.5,0.964602,0.868024
21,2.0,10.0,0.1,0.969912,0.867906
3,1.0,2.0,0.1,0.971681,0.867434
35,5.0,10.0,0.5,0.971681,0.867434
19,2.0,4.0,0.2,0.971681,0.867434
